In [1]:
#from transformers import AutoTokenizer
import json
import os
import random
random.seed(42)

In [2]:

def walk_directory(directory):
    l = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if not file.endswith(".csv"):
                l.append(os.path.join(root, file))
    print(f"Found {len(l)} files in {directory}")
    return l

def create_article_list(file):
    #print(f"Creating article list from {file}")
    articles = []
        # Load the JSON file
    with open(file, "r", encoding="utf-8") as fl:
        f = json.load(fl)
        articles.append(f["original_doc"])
        articles.append(f['articles']['gpt4o']['article'])
        articles.append(f['articles']['claude3.5sonnet']['article'])
        articles.append(f['articles']['llama3.1-405b']['article'])
        articles.append(f['articles']['qwen1.5-110b']['article'])
        #articles.append(f['articles']['watermarked']['article'])
    #print(f"Found {len(articles)} articles in {file}")
    return articles

def get_topic(path):
    topic =''
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        topic = data["topic"]
    return topic

In [3]:
import string
import re

def tokenize_article(path):
    word_distribution = {}
    topic = get_topic(path)
    for article in create_article_list(path):
        # Preprocess the article: strip, lowercase, remove punctuation
        article = article.strip().lower()
        # Replace punctuation with spaces
        article = re.sub(rf"[{re.escape(string.punctuation)}]", " ", article)
        # Split into words (simple whitespace split)
        words = article.split()
        for word in words:
            if word not in word_distribution:
                word_distribution[word] = {"count": 1, "topic": [topic]}
            else:
                word_distribution[word]["count"] += 1
                if topic not in word_distribution[word]["topic"]:
                    word_distribution[word]["topic"].append(topic)
    return word_distribution


def tokenize_txt(path):
    word_distribution = {}
    with open(path, "r", encoding="utf-8") as f:
        article = f.read().lower()
        # Make the article one line
        article = article.replace('\\n',' ').replace('\n', ' ').replace('\r', ' ')
        # Replace punctuation with spaces
        article = re.sub(rf"[{re.escape(string.punctuation)}]", " ", article)
        # Split into words (simple whitespace split)
        words = article.split()
        for word in words:
            if word not in word_distribution:
                word_distribution[word] = {"count": 1}#, "topic": [topic]}
            else:
                word_distribution[word]["count"] += 1
    return word_distribution


In [4]:
def create_distribution(piecelist,out_file):
    final_word_distribution = {}
    for file in piecelist:
        print(f"Processing file: {file}")
        #word_distribution = tokenize_article(file)
        word_distribution = tokenize_txt(file)
        # Merge the token distributions
        for item in word_distribution:
            if item not in final_word_distribution:
                final_word_distribution[item] = word_distribution[item]
            else:
                final_word_distribution[item]["count"] += word_distribution[item]["count"]
                '''for topic in word_distribution[item]["topic"]:
                    if topic not in final_word_distribution[item]["topic"]:
                        final_word_distribution[item]["topic"].append(topic)'''
    sorted_distribution = sorted(final_word_distribution.items(), key=lambda x: x[1]["count"], reverse=True)
    
    # Save the final token distribution to a JSON file
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(sorted_distribution, f, indent=4)

In [6]:
#create a for loop that loops through all subfolders of a selected folder
new_path = "./wmarked"

out_file = f'./json_files/new_tokenizer_tests/qwen_rag_watermarked.json'
os.remove(out_file) if os.path.exists(out_file) else None

piece = walk_directory(new_path)

token_distribution = create_distribution(piece,out_file)

Found 425 files in ./wmarked
Processing file: ./wmarked\result_divided_Company Policies_A_0.txt
Processing file: ./wmarked\result_divided_Company Policies_A_1.txt
Processing file: ./wmarked\result_divided_Company Policies_A_2.txt
Processing file: ./wmarked\result_divided_Company Policies_A_3.txt
Processing file: ./wmarked\result_divided_Company Policies_A_4.txt
Processing file: ./wmarked\result_divided_Company Policies_B_0.txt
Processing file: ./wmarked\result_divided_Company Policies_B_1.txt
Processing file: ./wmarked\result_divided_Company Policies_B_2.txt
Processing file: ./wmarked\result_divided_Company Policies_B_3.txt
Processing file: ./wmarked\result_divided_Company Policies_B_4.txt
Processing file: ./wmarked\result_divided_Company Policies_C_0.txt
Processing file: ./wmarked\result_divided_Company Policies_C_1.txt
Processing file: ./wmarked\result_divided_Company Policies_C_2.txt
Processing file: ./wmarked\result_divided_Company Policies_C_3.txt
Processing file: ./wmarked\result

In [ ]:
file_name = out_file

with open(file_name, 'r', encoding='utf-8') as f:
    token_data = json.load(f)

with open('./json_files/new_tokenizer_tests/unique_tokens.csv', 'w', encoding='utf-8') as csv_file:
    csv_file.write('topic,word\n')

for word, data in token_data:
    if data['count'] == 1:
        csv = f'{data["topic"][0]},{word}\n'
        with open('./json_files/new_tokenizer_tests/unique_tokens.csv', 'a', encoding='utf-8') as csv_file:
            csv_file.write(csv)


KeyError: 'topic'

In [5]:
import csv

# For each topic in selected_words, create a CSV file with the tokens and their counts in both CLEAN and WATERMARKED results
with open('./json_files/topic_list/selected_words.json', 'r', encoding='utf-8') as f:
    selected_words = json.load(f)
    topic_list = list(selected_words.keys())
    topic_list.append('More_common_words')

with open(f'./json_files/new_tokenizer_tests/qwen_rag_clean.json', 'r', encoding='utf-8') as f:
    clean_distribution = json.load(f)
    
with open(f'./json_files/new_tokenizer_tests/qwen_rag_watermarked.json', 'r', encoding='utf-8') as f:
    watermarked_distribution = json.load(f)



# Convert distributions to dicts for fast lookup
def dist_to_dict(distribution):
    return {token: data['count'] for token, data in distribution if token and isinstance(data, dict) and 'count' in data}

clean_dict = dist_to_dict(clean_distribution)
watermarked_dict = dist_to_dict(watermarked_distribution)

for topic in topic_list:
    tokens = selected_words[topic]['boosted_words'] if 'boosted_words' in selected_words[topic] else selected_words[topic]
    csv_filename = f'./json_files/new_tokenizer_tests/divided/{topic.replace(" ", "_").replace("/", "_").lower()}_token_counts.csv'
    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['token', 'clean_count', 'watermarked_count'])
        for token in tokens:
            clean_count = clean_dict.get(token, 0)
            watermarked_count = watermarked_dict.get(token, 0)
            writer.writerow([token, clean_count, watermarked_count])
    print(f'Saved: {csv_filename}')


Saved: ./json_files/new_tokenizer_tests/divided/company_policies_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/cybersecurity_news_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/incident_report_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_arts_and_culture_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_economy_and_market_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_education_systems_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_environmental_issues_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_health_and_wellness_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_news_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_politics_and_governance_token_counts.csv
Saved: ./json_files/new_tokenizer_tests/divided/local_sports_and_activities_token_counts.csv
Saved: ./json_files/new_tokenizer_tests

KeyError: 'More_common_words'

In [ ]:
'''#find all jsons that have the same topic and copy them to a new folder
import json
import os
import random
def copy_jsons_with_same_topic(source_dir, target_dir, topic):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if data.get("topic") == topic:
                        target_path = os.path.join(target_dir, file)
                        with open(target_path, "w", encoding="utf-8") as target_file:
                            json.dump(data, target_file, indent=4)
                        print(f"Copied {file} to {target_dir}")
source_directory = "./distribution_clean"
target_directory = "./distribution_divided"
with open("./json_files/topic_list/selected_words.json", "r") as f:
    selected_topics = json.load(f)
    list_of_topics = list(selected_topics.keys())

for topic_to_copy in list_of_topics:
    string_path = '/' + topic_to_copy.replace(" ", "_").replace("/", "_").lower()
    copy_jsons_with_same_topic(source_directory, target_directory +string_path, topic_to_copy)
    print(f"All JSON files with topic '{topic_to_copy}' have been copied to {target_directory}.")'''

KeyboardInterrupt: 

In [18]:
'''import glob

def remove_watermarked_from_jsons(path):
    """
    Remove all 'watermarked' elements from JSON files in the given directory (recursively).
    Modifies files in place.
    """
    json_files = glob.glob(os.path.join(path, '**', '*.json'), recursive=True)
    for file in json_files:
        with open(file, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except Exception as e:
                print(f"Error reading {file}: {e}")
                continue
        changed = False
        # Remove 'watermarked' key if present at the top level
        if 'watermarked' in data:
            del data['watermarked']
            changed = True
        # Remove 'watermarked' from nested 'articles' if present
        if 'articles' in data and isinstance(data['articles'], dict):
            if 'watermarked' in data['articles']:
                del data['articles']['watermarked']
                changed = True
        if changed:
            with open(file, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
                print(f"Updated: {file}")


remove_watermarked_from_jsons("./distribution_same_topic_cyber_words")'''

Updated: ./distribution_same_topic_cyber_words\0050.json
Updated: ./distribution_same_topic_cyber_words\0060.json
Updated: ./distribution_same_topic_cyber_words\0079.json
Updated: ./distribution_same_topic_cyber_words\0104.json
Updated: ./distribution_same_topic_cyber_words\0120.json
Updated: ./distribution_same_topic_cyber_words\0123.json
Updated: ./distribution_same_topic_cyber_words\0141.json
Updated: ./distribution_same_topic_cyber_words\0194.json
Updated: ./distribution_same_topic_cyber_words\0205.json
Updated: ./distribution_same_topic_cyber_words\0212.json
Updated: ./distribution_same_topic_cyber_words\0221.json
Updated: ./distribution_same_topic_cyber_words\0237.json
Updated: ./distribution_same_topic_cyber_words\0276.json
Updated: ./distribution_same_topic_cyber_words\0292.json
Updated: ./distribution_same_topic_cyber_words\0293.json
Updated: ./distribution_same_topic_cyber_words\0352.json
Updated: ./distribution_same_topic_cyber_words\0390.json
Updated: ./distribution_same_to

In [9]:
clean_json = './json_files/new_tokenizer_tests/inverse/tokens_wmarkdataset_samefacts_inverse_CLEAN.json'
watermarked_json = './json_files/new_tokenizer_tests/word_dist_wmarked.json'

with open(clean_json, 'r', encoding='utf-8') as f:
    clean_data = json.load(f)
with open(watermarked_json, 'r', encoding='utf-8') as f:
    watermarked_data = json.load(f)

#for each token in clean_data with count == 1, check if it is in watermarked_data and print its count
unique_tokens = {}
for token, data in clean_data:
    if data['count'] == 1:
        unique_tokens[token] = {
            'clean_count': data['count'],
            'watermarked_count': 0
        }
        for w_token, w_data in watermarked_data:
            if w_token == token:
                unique_tokens[token]['watermarked_count'] = w_data['count']
                break
print(f"Found {len(unique_tokens)} unique tokens in CLEAN data.")
with open('./json_files/new_tokenizer_tests/inverse/unique_tokens_word_dist.csv', 'w', encoding='utf-8') as csv_file:
    csv_file.write('token,clean_count,watermarked_count\n')
    for token, counts in unique_tokens.items():
        csv = f'{token},{counts["clean_count"]},{counts["watermarked_count"]}\n'
        csv_file.write(csv)

Found 27373 unique tokens in CLEAN data.
